In [1]:
import json
import numpy as np
import pandas as pd

array = np.load("../embeddings/image_embeddings.npy")

new_arr = array[3]

In [2]:
user_pref = new_arr.reshape(1,512)

In [3]:
np.save("../user_vector_example.npy", user_pref)

In [19]:
user_pref

array([[ 1.36320395e-02,  1.19571369e-02, -5.20074042e-03,
         1.89508703e-02,  2.16959771e-02, -3.23209800e-02,
         3.20345126e-02,  3.69977765e-02, -2.45844536e-02,
         6.14133431e-04,  7.31317848e-02,  1.58170406e-02,
        -5.15028760e-02, -3.02967243e-02,  3.46830562e-02,
        -2.12663449e-02, -7.37836510e-02,  3.05352006e-02,
         6.56093433e-02,  1.76298022e-02,  1.18891895e-03,
         1.46504743e-02, -1.10739404e-02, -3.14447805e-02,
         4.40867152e-03,  1.22615267e-02,  3.26562935e-04,
        -4.06591967e-02, -3.53504904e-02,  1.24681238e-02,
        -3.36626603e-04,  6.02391362e-02, -7.78508978e-03,
        -2.73837633e-02,  4.56933677e-03,  1.94908939e-02,
         5.17463032e-03, -7.19212042e-03, -3.92085314e-02,
         3.08702085e-02,  2.60334671e-03, -1.13090985e-02,
        -1.85042713e-02, -1.36185512e-02,  1.58220588e-03,
        -6.56871945e-02, -9.83968191e-03,  1.45037891e-02,
        -7.82237481e-03, -1.58834606e-02,  4.58628051e-0

In [22]:
df_top5.head()

,rank,index,phrase,cosine
0,1,41,"{'column': 'Vibes', 'row_idx': 41, 'text': 'a ...",0.272206
1,2,121,"{'column': 'Destination Features', 'row_idx': ...",0.267934
2,3,10,"{'column': 'Vibes', 'row_idx': 10, 'text': 'a ...",0.267689
3,4,1,"{'column': 'Vibes', 'row_idx': 1, 'text': 'a t...",0.266610
4,5,2,"{'column': 'Vibes', 'row_idx': 2, 'text': 'a t...",0.265372


In [ ]:
import numpy as np
import pandas as pd
import json

KW_MAT_PATH  = "../embeddings/keywords_embeddings.npy"
PHRASES_JSON = "../json_files/keywords_index.json"
TOP_K = 5

def l2_normalize_rows(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    if X.ndim == 1:
        n = np.linalg.norm(X)
        return (X / max(n, eps))[None, :]
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, eps)

def load_phrase_texts(json_path: str) -> list | None:
    """
    Returns a list where out[i] is the phrase TEXT for keyword row i.
    Prioritizes 'text' fields if present. Supported shapes include:
      - [{"text": "..."}], or [{"index": i, "text": "..."}, ...]
      - {"keywords": [{"text": "..."} , ...]}
      - {"0": {"text": "..."}, "1": {"text": "..."}, ...}
      - ["raw string", ...]  (falls back to raw strings)
      - {"0": "raw string", "1": "raw string", ...}
    """
    try:
        with open(json_path, "r") as f:
            data = json.load(f)
    except FileNotFoundError:
        return None

    def list_from_indexed_dict(d, value_key=None):
        items = sorted(((int(k), v) for k, v in d.items()), key=lambda t: t[0])
        if value_key:
            return [v.get(value_key) for _, v in items]
        return [v for _, v in items]

    # Case: list[...] (either dicts with 'text' or plain strings)
    if isinstance(data, list):
        if all(isinstance(x, dict) for x in data):
            # If explicit indices exist, respect them; else keep order
            if all('index' in x for x in data) and all('text' in x for x in data):
                max_idx = max(int(x['index']) for x in data)
                out = [None] * (max_idx + 1)
                for x in data:
                    out[int(x['index'])] = x['text']
                return out
            # ordered list of dicts -> take 'text' if present, else try common fallbacks
            key = 'text' if all('text' in x for x in data) else (
                'phrase' if all('phrase' in x for x in data) else (
                'keyword' if all('keyword' in x for x in data) else None))
            if key:
                return [x[key] for x in data]
        elif all(isinstance(x, str) for x in data):
            return data

    # Case: {"keywords": [...]}
    if isinstance(data, dict) and isinstance(data.get("keywords"), list):
        kw = data["keywords"]
        if all(isinstance(x, dict) and 'text' in x for x in kw):
            return [x['text'] for x in kw]
        if all(isinstance(x, str) for x in kw):
            return kw

    # Case: {"0": {...}, "1": {...}} with 'text' inside
    if isinstance(data, dict) and all(isinstance(k, str) and k.isdigit() for k in data.keys()):
        if all(isinstance(v, dict) and 'text' in v for v in data.values()):
            return list_from_indexed_dict(data, value_key='text')
        if all(isinstance(v, str) for v in data.values()):
            return list_from_indexed_dict(data, value_key=None)

    # Fallbacks that match earlier patterns (phrase strings)
    if isinstance(data, dict) and isinstance(data.get("index_to_phrase"), dict):
        d = data["index_to_phrase"]
        if all(isinstance(v, dict) and 'text' in v for v in d.values()):
            return list_from_indexed_dict(d, value_key='text')
        if all(isinstance(v, str) for v in d.values()):
            return list_from_indexed_dict(d, value_key=None)

    return None

# --- Use your in-memory user vector ---
# You said you already have user_pref with shape (1,512). If it's (512,), this still works.
user_vec = l2_normalize_rows(user_pref)                 # (1, 512)

# --- Keyword matrix ---
kw_mat  = np.load(KW_MAT_PATH).astype(np.float32)       # or: kw_mat = np.asarray(kw_mat, dtype=np.float32)
kw_norm = l2_normalize_rows(kw_mat)                     # (M, 512)

# --- Similarity & Top-K ---
sims = (user_vec @ kw_norm.T).ravel()                   # (M,)
k = min(TOP_K, sims.size)
topk_idx = np.argpartition(sims, -k)[-k:]
topk_idx = topk_idx[np.argsort(sims[topk_idx])[::-1]]
topk_scores = sims[topk_idx]

# --- Load phrase TEXTs and build DataFrame ---
phrase_texts = load_phrase_texts(PHRASES_JSON)  # list[str] aligned to kw_mat rows (if provided)

df_top5 = pd.DataFrame({
    "rank":   np.arange(1, k+1, dtype=int),
    "index":  topk_idx.astype(int),
    "Phrase": [phrase_texts[i] if (phrase_texts and i < len(phrase_texts)) else None for i in topk_idx],
    "cosine": topk_scores.astype(float),
})

print(df_top5)


   rank  index                                   Phrase    cosine
0     1     41  a travel scene that feels contemplative  0.279630
1     2     31     a travel scene that feels backpacker  0.275961
2     3    111                  a scene of a surf break  0.274998
3     4     50         a travel scene that feels sombre  0.274043
4     5     10       a travel scene that feels secluded  0.272023


In [ ]:
import numpy as np
import pandas as pd
import json

# ---- Fixed inputs ----
KW_MAT_PATH  = "../embeddings/keywords_embeddings.npy"
PHRASES_JSON = "../json_files/keywords_index.json"
TOP_K = 5

def l2_normalize_rows(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    if X.ndim == 1:
        n = np.linalg.norm(X)
        return (X / max(n, eps))[None, :]
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, eps)

# ---- Load embeddings and phrases ----
kw_mat  = np.load(KW_MAT_PATH).astype(np.float32)      # shape: (M, 512)
kw_norm = l2_normalize_rows(kw_mat)

with open(PHRASES_JSON, "r") as f:
    data = json.load(f)                                 # list of dicts
phrase_texts = [x["text"] for x in data]                # strictly use "text"
# Optional sanity check (length only)
assert len(phrase_texts) == kw_mat.shape[0], "JSON/texts length must match embedding rows."

# ---- User vector already in memory ----
user_vec = l2_normalize_rows(user_pref)                 # (1, 512) or (512,)

# ---- Similarity & Top-K ----
sims = (user_vec @ kw_norm.T).ravel()                   # (M,)
k = min(TOP_K, sims.size)
topk_idx = np.argpartition(sims, -k)[-k:]
topk_idx = topk_idx[np.argsort(sims[topk_idx])[::-1]]
topk_scores = sims[topk_idx]

# ---- Final DataFrame ----
df_topk = pd.DataFrame({
    "rank":   np.arange(1, k+1, dtype=int),
    "index":  topk_idx.astype(int),
    "Phrase": [phrase_texts[i] for i in topk_idx],      # strictly 'text'
    "cosine": topk_scores.astype(float),
})

print(df_topk)


   rank  index                                             Phrase    cosine
0     1     41            a travel scene that feels contemplative  0.263302
1     2     84  a place with rough roads and scenic views wher...  0.261739
2     3      2             a travel scene that feels feels serene  0.260903
3     4     66  a place with rough trails could do mountain bi...  0.260727
4     5     13                    a travel scene that feels quiet  0.260386
